# 04 — Validação Out-of-Sample (Holdout)

**Objetivo:** verificar se o modelo generaliza para dados que nunca foram usados no ajuste.

**Por que isso importa?**  
Um modelo com R² alto no treino pode simplesmente ter "decorado" a série (overfitting). Reservar as últimas 24 semanas — que nunca foram usadas para calibrar λ, γ ou κ — é o único jeito de saber se o modelo capturou padrões reais ou ruído.

Roteiro:
1. Re-ajustar o modelo no período de treino (sem ver o holdout)
2. Gerar previsões semana a semana no holdout
3. Calcular métricas: R², MAPE, RMSE
4. Visualização: fitted vs. actual — treino e holdout
5. Análise de erro por semana
6. Relatório final de validação

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from src.model import MMMModel, DEFAULT_CHANNEL_PARAMS
from src.validation import (
    temporal_split,
    compute_metrics,
    holdout_week_by_week,
    plot_holdout,
    validation_report,
    durbin_watson,
)

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.dpi"] = 110
print("Módulos carregados.")

## 1. Carregar Dados e Re-ajustar o Modelo

In [ ]:
df = pd.read_csv("../data/MMM_Synth_Weekly_Data.csv", parse_dates=["week"])
df = df.sort_values("week").reset_index(drop=True)

HOLDOUT_WEEKS = 24
df_train, df_holdout = temporal_split(df, holdout_weeks=HOLDOUT_WEEKS)

print(f"Treino  : {len(df_train)} semanas  "
      f"({df_train['week'].min().date()} → {df_train['week'].max().date()})")
print(f"Holdout : {len(df_holdout)} semanas  "
      f"({df_holdout['week'].min().date()} → {df_holdout['week'].max().date()})")

In [ ]:
# Ajustar SOMENTE no treino — holdout nunca é visto pelo modelo
mmm = MMMModel(channel_params=DEFAULT_CHANNEL_PARAMS, hac_lags=4)
mmm.fit(df_train)

train_metrics = mmm.metrics()
print(f"Treino — R²: {train_metrics['R2']:.4f} | MAPE: {train_metrics['MAPE_%']:.2f}%")

## 2. Previsão Semana a Semana no Holdout

In [ ]:
holdout_pred = holdout_week_by_week(mmm, df_holdout)

# Métricas out-of-sample
hold_metrics = compute_metrics(
    holdout_pred["actual"].values,
    holdout_pred["predicted"].values,
    n_params=len(mmm.feature_cols_),
)

print("=" * 55)
print("MÉTRICAS — HOLDOUT (Out-of-Sample)")
print("=" * 55)
print(f"  R²     : {hold_metrics['R2']:.4f}  ({hold_metrics['R2']*100:.1f}%)")
print(f"  MAPE   : {hold_metrics['MAPE_pct']:.2f}%")
print(f"  RMSE   : {hold_metrics['RMSE']:,.0f} assinantes/semana")
print(f"  MAE    : {hold_metrics['MAE']:,.0f} assinantes/semana")
print(f"  N obs  : {hold_metrics['n_obs']}")

holdout_pred.head(10)

## 3. Visualização — Treino e Holdout

In [ ]:
fig = plot_holdout(
    holdout_df=holdout_pred,
    train_actual=df_train["new_subscribers"].values,
    train_fitted=mmm.result_.fittedvalues,
    title="MMM Video+ — Validação Out-of-Sample (Holdout Jul–Dez 2024)",
    save_path="../outputs/holdout_validation.png",
)
plt.show()

## 4. Análise de Erro por Semana

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribuição do erro %
axes[0].hist(holdout_pred["abs_pct_error"], bins=12,
             color="#0f3460", alpha=0.8, edgecolor="white")
axes[0].axvline(hold_metrics["MAPE_pct"], color="#e94560", lw=1.5,
                linestyle="--", label=f"MAPE = {hold_metrics['MAPE_pct']:.2f}%")
axes[0].set_xlabel("Erro Absoluto (%)")
axes[0].set_ylabel("Frequência")
axes[0].set_title("Distribuição do Erro % — Holdout", fontweight="bold")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Scatter: previsto × real
axes[1].scatter(
    holdout_pred["actual"], holdout_pred["predicted"],
    color="#0f3460", alpha=0.7, edgecolors="white", linewidths=0.5
)
lims = [
    min(holdout_pred[["actual", "predicted"]].min()),
    max(holdout_pred[["actual", "predicted"]].max()),
]
axes[1].plot(lims, lims, color="#e94560", lw=1.5, linestyle="--", label="Linha perfeita")
axes[1].set_xlabel("Real (assinantes)")
axes[1].set_ylabel("Previsto (assinantes)")
axes[1].set_title("Real vs. Previsto — Holdout", fontweight="bold")
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))

plt.tight_layout()
plt.savefig("../outputs/holdout_error_analysis.png", dpi=130, bbox_inches="tight")
plt.show()

# Pior semana
worst = holdout_pred.loc[holdout_pred["abs_pct_error"].idxmax()]
print(f"Pior semana: erro de {worst['abs_pct_error']:.2f}%")
print(f"  Real: {worst['actual']:,.0f} | Previsto: {worst['predicted']:,.0f}")

## 5. Relatório Final de Validação

In [ ]:
report = validation_report(mmm, df_train, df_holdout)

print("=" * 60)
print("RELATÓRIO DE VALIDAÇÃO — MMM Video+ v2")
print("=" * 60)

print(f"\nPERÍODO DE TREINO ({report['n_train']} semanas)")
for k, v in report["train"].items():
    print(f"  {k:15s}: {v}")

print(f"\nPERÍODO DE HOLDOUT ({report['n_holdout']} semanas — out-of-sample)")
for k, v in report["holdout"].items():
    print(f"  {k:15s}: {v}")

print(f"\nDurbin-Watson (resíduos de treino): {report['durbin_watson']}")
print("  (DW < 1.5 → autocorrelação positiva → HAC é necessário)")

# Comparação resumida
print("\n" + "-" * 60)
print(f"{'Métrica':20s} {'Treino':>12s} {'Holdout':>12s}")
print("-" * 60)
for k in ["R2", "MAPE_pct", "RMSE"]:
    tk = report["train"].get(k, "—")
    hk = report["holdout"].get(k, "—")
    print(f"  {k:18s} {str(tk):>12s} {str(hk):>12s}")
print("-" * 60)

## 6. Salvar Previsão do Holdout

In [ ]:
out_path = "../data/MMM_Holdout_Predictions.csv"
holdout_pred.to_csv(out_path, index=False)
print(f"Previsão do holdout salva: {out_path}")

print("""
═══════════════════════════════════════════════════
 Pipeline MMM Video+ concluído com sucesso!
═══════════════════════════════════════════════════

 Arquivos gerados em /outputs:
   - channel_significance.png
   - cpa_implied_vs_assumed.png
   - holdout_validation.png
   - holdout_error_analysis.png
   - adstock_decay_example.png
   - hill_saturation_sensitivity.png
   - saturation_curves_by_channel.png
   - eda_subscribers_over_time.png
   - eda_spend_by_channel.png
   - eda_seasonality_trend.png

 Próximos passos:
   → Calibrar com dados reais de spend
   → Migrar para framework Bayesiano (PyMC-Marketing / Robyn)
   → Geo-experiments para calibração incremental
   → Ortogonalizar variável de tendência (VIF trend alto)
""")